# Install of Libraries

In [1]:
# Install required libraries

!pip install -q pandas numpy sentence-transformers faiss-cpu transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 89.9 MB/s eta 0:00:00


In [24]:
!pip install -q google-genai

# Necessary Libraries

In [34]:
import pandas as pd
import numpy as np
import torch
import faiss
import time
from google.genai.errors import ServerError

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

Pandas: 2.2.3
NumPy: 2.1.3
PyTorch: 2.11.0+cu130
CUDA Available: True


# Dataset Loading

In [3]:
df = pd.read_csv("case_files_total.csv")

print("Dataset Shape:", df.shape)
df.head(6)

Dataset Shape: (53446, 10)


,Unnamed: 0,name,case_category,case_type,case_info,judgement,tokens,sentences,label,proof_sentence
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,74634371.txt,civil,writ petition,reportable in the supreme court of india civil...,thus spake benjamin franklin in his letter of ...,9974.0,254.0,Accepted,"34. we are, therefore, of the view that this a..."
2,2,746230.txt,civil,writ petition,case number appeal civil 588 of 1972 petitio...,difficult to share the view taken by the high...,1411.0,56.0,Accepted,"6. accordingly, we allow the appeal with costs."
3,3,746330.txt,civil,writ petition,petitioner western india plywood limited vs. r...,j u d g m e n t the following judgement of the...,2616.0,85.0,Accepted,for the aforesaid reasons this appeal is allowed.
4,4,746327.txt,civil,writ petition,"petitioner asstt.\ncustodian, e.p. ors.\nvs. r...",custodian first stated that the property was n...,2763.0,95.0,Accepted,appeal allowed.
5,5,746119.txt,criminal,special leave petition,"judgment c.a. vaidialingam, j. 1.\n",this appeal by special leave is against the ju...,2571.0,106.0,Rejected,15. in the result the appeal fails and is dism...


In [4]:
print("Rows:", len(df))
print("Columns:")

for col in df.columns:
    print("-", col)

Rows: 53446
Columns:
- Unnamed: 0
- name
- case_category
- case_type
- case_info
- judgement
- tokens
- sentences
- label
- proof_sentence


# Data Preprocessing

In [5]:
# Checking missing values
df.isnull().sum()

,0
Unnamed: 0,0
name,1
case_category,9834
case_type,736
case_info,25
judgement,3302
tokens,2
sentences,2
label,2
proof_sentence,2


In [6]:

# HANDLE MISSING VALUES

# Numerical columns → Median
numeric_columns = df.select_dtypes(
    include=[np.number]
).columns

for col in numeric_columns:
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)

        print(
            f"{col}: Missing values replaced with median = {median_value}"
        )


# Text / categorical columns → Mode
text_columns = df.select_dtypes(
    include=["object", "string", "category"]
).columns

for col in text_columns:
    if df[col].isnull().sum() > 0:

        # Most frequently occurring value
        mode_value = df[col].mode()[0]

        df[col] = df[col].fillna(mode_value)

        print(
            f"{col}: Missing values replaced with mode = {mode_value}"
        )


# Check remaining missing values
print("\nRemaining Missing Values:")
print(df.isnull().sum())

tokens: Missing values replaced with median = 2400.0
sentences: Missing values replaced with median = 76.0
name: Missing values replaced with mode = 1077544.txt
case_category: Missing values replaced with mode = civil
case_type: Missing values replaced with mode = appeal
case_info: Missing values replaced with mode = order 1.

judgement: Missing values replaced with mode = respondents judgment dr. arijit pasayat, j. 1. challenge in this appeal is to the judgment passed by the appellate tribunal for electricity, new delhi  hereinafter referred to as to the tribunal .
the appeal has been filed under section 125 of the electricity act, 2003  in short the act .
2. the primary stand of the appellant is that though the tribunal accepted that electricity regulatory commissions  in short the regulatory commission  did not have any power to determine tariff for trading, it invoked sections 60 and 66 of the act to direct all regulatory commissions to fix trading margins as if it involved tariff 

# Preparing dataset for RAG

In [7]:
# Keep only rows having judgment text

df = df.dropna(subset=["judgement"])

# Remove duplicate judgments

df = df.drop_duplicates(subset=["judgement"])

print("Dataset after cleaning:", df.shape)

Dataset after cleaning: (41007, 10)


In [8]:
df["text_length"] = df["judgement"].astype(str).str.len()

print(df["text_length"].describe())

count    41007.000000
mean     15682.937377
std      10733.611932
min          1.000000
25%       6237.000000
50%      13444.000000
75%      24822.500000
max      32765.000000
Name: text_length, dtype: float64


In [9]:
RAG_DOCS = 10000

rag_df = df.sample(
    n=min(RAG_DOCS, len(df)),
    random_state=42
).reset_index(drop=True)

print("RAG Documents:", len(rag_df))

RAG Documents: 10000


In [10]:
# Create Document Text
def create_document(row):

    return f"""
Case Category: {row.get('case_category', 'Unknown')}

Case Type: {row.get('case_type', 'Unknown')}

Case Information:
{row.get('case_info', 'Unknown')}

Judgment:
{row['judgement']}
""".strip()


rag_df["document"] = rag_df.apply(create_document, axis=1)

print(rag_df["document"].iloc[0][:3000])

Case Category: civil

Case Type: writ petition

Case Information:
petitioner h. a. k. rao, chartered accountant vs. respondent council of institute of chartered accountants of india, new date of judgment 13/12/1966 bench rao, k. subba  cj  bench rao, k. subba  cj  shah, j.c. sikri, s.m. ramaswami, v. vaidyialingam, c.a. citation 1967 air 1257 1967 scr  2  256 act chartered accountants act 1949--clause  ii , part ii, second schedule--councils power to specify any act or omission other than those covered by act or regulations as professional misconduct--scope of.
headnote the council of the institute of chartered accountants, by a notification issued on february 22, 1964, in exercise of powers conferred by cl.  
ii  of part 11 of the second schedule to the chartered accountants act, 1949, notified, inter alia, that a member of the institute shall be deemed to be guilty of misconduct if, in connection with elections to the councils of the lnstitute.,
he took part directly or indirectly in

In [11]:
# Text Chunking
def chunk_text(text, chunk_size=700, overlap=100):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [12]:
sample_chunks = chunk_text(rag_df["document"].iloc[0])

print("Number of chunks:", len(sample_chunks))

print("\nFirst chunk:\n")
print(sample_chunks[0][:2000])

Number of chunks: 7

First chunk:

Case Category: civil Case Type: writ petition Case Information: petitioner h. a. k. rao, chartered accountant vs. respondent council of institute of chartered accountants of india, new date of judgment 13/12/1966 bench rao, k. subba cj bench rao, k. subba cj shah, j.c. sikri, s.m. ramaswami, v. vaidyialingam, c.a. citation 1967 air 1257 1967 scr 2 256 act chartered accountants act 1949--clause ii , part ii, second schedule--councils power to specify any act or omission other than those covered by act or regulations as professional misconduct--scope of. headnote the council of the institute of chartered accountants, by a notification issued on february 22, 1964, in exercise of powers conferred by cl. ii of part 11 of the second schedule to the chartered accountants act, 1949, notified, inter alia, that a member of the institute shall be deemed to be guilty of misconduct if, in connection with elections to the councils of the lnstitute., he took part di

In [13]:
# Create Chunk Database
documents = []

for idx, row in rag_df.iterrows():

    chunks = chunk_text(row["document"])

    for chunk_id, chunk in enumerate(chunks):

        documents.append({
            "case_id": idx,
            "chunk_id": chunk_id,
            "case_category": row.get("case_category", "Unknown"),
            "case_type": row.get("case_type", "Unknown"),
            "case_info": row.get("case_info", "Unknown"),
            "text": chunk
        })

print("Total chunks:", len(documents))

Total chunks: 53761


In [14]:
chunks_df = pd.DataFrame(documents)

chunks_df.head(6)

,case_id,chunk_id,case_category,case_type,case_info,text
0,0,0,civil,writ petition,"petitioner h. a. k. rao, chartered accountant ...",Case Category: civil Case Type: writ petition ...
1,0,1,civil,writ petition,"petitioner h. a. k. rao, chartered accountant ...",or the regulations. 263 f while the regulation...
2,0,2,civil,writ petition,"petitioner h. a. k. rao, chartered accountant ...","or through any other person, in any of the fol..."
3,0,3,civil,writ petition,"petitioner h. a. k. rao, chartered accountant ...",carrying out the objects of the act. the centr...
4,0,4,civil,writ petition,"petitioner h. a. k. rao, chartered accountant ...",of a candidate within a distance of 200 meters...
5,0,5,civil,writ petition,"petitioner h. a. k. rao, chartered accountant ...",conflict with the same. the further argument w...


# Load Embedding Model

In [15]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Device:", device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Device: cuda


In [16]:
# Generate Embeddings
texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/841 [00:00<?, ?it/s]

Embedding shape: (53761, 384)


In [17]:
# Create FAISS Vector Database
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings.astype("float32"))

print("FAISS index created.")
print("Total vectors:", index.ntotal)

FAISS index created.
Total vectors: 53761


In [18]:
# Save FAISS Index
faiss.write_index(
    index,
    "legal_judgment_faiss.index"
)

print("FAISS index saved.")

# Save chunk metadata
chunks_df.to_pickle(
    "legal_judgment_chunks.pkl"
)

print("Chunk metadata saved.")

FAISS index saved.
Chunk metadata saved.


In [19]:
# Build Retrieval Function
def retrieve_documents(query, top_k=5):

    # Convert query into embedding
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    # Search FAISS
    scores, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        result = chunks_df.iloc[idx].to_dict()

        result["similarity_score"] = float(score)

        results.append(result)

    return results

In [20]:
# Test Semantic Search
query = "What was the main issue considered by the court?"

results = retrieve_documents(query, top_k=5)

for i, result in enumerate(results):

    print("=" * 80)

    print("Result:", i + 1)

    print("Similarity:",
          round(result["similarity_score"], 4))

    print("Case Type:",
          result["case_type"])

    print("\nText:")
    print(result["text"][:1500])

Result: 1
Similarity: 0.639
Case Type: appeal

Text:
parties. in fact, the agreement was not acted upon. the high court proceeded on an erroneous basis as if some of the issues were not pressed before the trial court and the high court. the clauses of the agreement on which the trial court and the high court placed reliance do not prove the essence of the transactions and or intention and should not have been given undue importance. some of the basic issues like issue number12 were not adjudicated by the trial court and the high court. though reference was placed on the objections filed to the application under section 145 of the cr.p.c., stand of the appellant was not taken note of. in fact, an application had been filed for taking note of the objections which unfortunately the high court treated to have become infructuous as it was listed on the day the judgment was delivered. while considering a plea that the agreement was not intended to be acted upon, veil has to be lifted by cons

In [21]:
query = "What was the final decision of the court?"

results = retrieve_documents(query, top_k=5)

for i, result in enumerate(results):

    print("=" * 80)

    print("Result:", i + 1)
    print("Similarity:", result["similarity_score"])
    print(result["text"][:1000])

Result: 1
Similarity: 0.6803998947143555
and which had become final either because no appeal was carried to this court or, if carried, the same was dismissed. this benefit would be available to no other.
Result: 2
Similarity: 0.6737040281295776
for the parties, we record that all of them at the close of the hearing of these cases conceded that the jurisdiction of this court under article 32 of the constitution cannot be invoked to challenge the validity of a final judgment order passed by this court after exhausting the remedy of review under article 137 of the constitution read with order xl rule 1 of the supreme court rules 1966. however, all the learned counsel for the parties as also the learned attorney-general who appeared as amicus curiae, on the notice of this court, adopted an unusual unanimous approach to plead that even after exhausting the remedy of review under article 137 of the constitution, an aggrieved person might be provided with an opportunity under inherent powers 

In [22]:
# RAG Context
def build_context(results):

    context_parts = []

    for i, result in enumerate(results):

        context_parts.append(
            f"""
SOURCE {i+1}

Case Category:
{result['case_category']}

Case Type:
{result['case_type']}

Case Information:
{result['case_info']}

Relevant Judgment:
{result['text']}
"""
        )

    return "\n".join(context_parts)

In [23]:
results = retrieve_documents(
    "What was the main issue in this case?",
    top_k=3
)

context = build_context(results)

print(context[:5000])


SOURCE 1

Case Category:
civil

Case Type:
election petition

Case Information:

supreme court of india
harmohinder singh pradhan vs ranjeet singh talwandi ors on 28 april, 2005author r lahoti
bench cji r.c. lahoti, g.p. mathur, p.k. balasubramanyan
case number
appeal  civil  8315 of 2003

petitioner
harmohinder singh pradhan

respondent
ranjeet singh talwandi ors

date of judgment 28/04/2005

bench
cji r.c. lahoti, g.p. mathur p.k. balasubramanyan

judgment
j u
d g m e n t

r.c. lahoti, cji

this appeal under section 116a of the representation of 


Relevant Judgment:
cause of action under clause a of rule 11 of order vii of the code. no fault can be found with the view taken by the learned designated election judge of the high court. the appeal is devoid of any merit and is dismissed with costs.


SOURCE 2

Case Category:
civil

Case Type:
writ petition

Case Information:
reportable in the supreme court of india civilappellate jurisdiction civil appeal number 6905 of 2005 m s. s.k.l

In [25]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("Gemini API key loaded:", GEMINI_API_KEY is not None)

Gemini API key loaded: True


In [38]:
# Gemini client
from google import genai

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client ready.")

Gemini client ready.


# RAG Prompt

In [28]:
def create_prompt(question, context):

    prompt = f"""
You are a legal document retrieval assistant.

Answer the user's question using ONLY the provided
retrieved legal judgment context.

Do not invent legal facts.

If the retrieved context does not contain enough
information to answer the question, clearly say:

"Insufficient information in the retrieved documents."

Provide a concise and well-structured answer.

User Question:
{question}

Retrieved Legal Context:
{context}

Answer:
"""

    return prompt

In [42]:
from google.genai.errors import ServerError
import time


def generate_answer(question, top_k=5):

    # Retrieve relevant documents
    results = retrieve_documents(
        question,
        top_k=top_k
    )

    # No relevant documents found
    if len(results) == 0:
        return (
            "Insufficient information in the retrieved documents.",
            []
        )

    # Build context
    context = build_context(results)

    # Create RAG prompt
    prompt = create_prompt(
        question,
        context
    )

    # Try multiple Gemini models

    models = [
        "gemini-3.5-flash",
        "gemini-3.5-flash-lite",
        "gemini-2.5-flash-lite"
    ]

    for model_name in models:

        try:

            print(f"Trying model: {model_name}")

            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            print(f"Successfully generated using: {model_name}")

            return response.text, results

        except ServerError:

            print(
                f"{model_name} is temporarily unavailable. "
                f"Trying next model..."
            )

            time.sleep(2)


    # If all models fail


    return (
        "Gemini models are temporarily unavailable. "
        "Please try again after some time.",
        results
    )

In [40]:
question = """
What was the main legal issue considered by the court?
"""

answer, sources = generate_answer(
    question,
    top_k=5
)

print(answer)

Based on the retrieved legal documents, the main legal issues considered by the courts in the respective cases are as follows:

*   **In *C.N. Ramappa Gowda v. C.C. Chandregowda* (Source 2):** 
    1. Whether the High Court exceeded its jurisdiction by ordering a retrial and allowing the defendants to file a written statement and documents without legally sustainable reasons, despite prior multiple opportunities.
    2. Whether a trial court can straightway pass a decree in favor of a plaintiff when the defendant fails to file a written statement, or if it must still try the suit on its merits by critically appreciating the plaintiff's evidence.

*   **In *Chandrasingh Manibhai v. Surjit Lal Ladhamal Chhabda* (Source 5):** 
    Whether the Bombay Rents, Hotel and Lodging House Rates Control Act, LVII of 1947, applies retrospectively to pending appeals or if its retrospective effect is limited only to the cases specified in Section 50 of the Act.

*   **In *Krishan Lal v. State of Jammu

In [44]:
question = """
What was the final decision of the court?
"""

results = retrieve_documents(
    question,
    top_k=10
)

print("Number of retrieved documents:", len(results))

for i, result in enumerate(results):

    print("\n" + "=" * 100)
    print(f"SOURCE {i + 1}")
    print("=" * 100)

    print("Similarity:",
          round(result["similarity_score"], 4))

    print("Case Category:",
          result["case_category"])

    print("Case Type:",
          result["case_type"])

    print("Case Information:",
          result["case_info"])

    print("\nRetrieved Text:")
    print(result["text"][:2000])

Number of retrieved documents: 10

SOURCE 1
Similarity: 0.6804
Case Category: civil
Case Type: appeal
Case Information: petitioner mamad hassam bhagad and others vs. respondent state of gujarat and others date of judgment 09/05/1996 bench venkataswami k.  j  bench venkataswami k.  j  anand, a.s.  
j  citation 1996 air 2057 jt 1996  5  327 1996 scale  4 313 act headnote judgment j u d g m e n t k. venkataswami,j. this appeal preferred under section 19 of the terrorist and disruptive activities  prevention  act.
1987  hereinafter referred to as tada  challenges the judgment and order of the learned designated judge  tada , jamnagar dated july 1, 1994.
the learned designated judge confirmed an order of attachment of seven vessels ships under section 7a of tada pending further investigation in the matter.
at the outset we would like to make it clear that we do not propose to deal with the matter either in detail or to give a final decision in this matter in view of the fact that when the j

In [45]:
question = """
What was the final decision or outcome of the court?
What did the court ultimately hold or order?
Was the case allowed, dismissed, disposed of, or otherwise decided?
"""

results = retrieve_documents(
    question,
    top_k=10
)

for i, result in enumerate(results):

    print("\n" + "=" * 100)

    print(f"SOURCE {i + 1}")
    print("Similarity:",
          round(result["similarity_score"], 4))

    print("Case Information:",
          result["case_info"])

    print("\nText:")
    print(result["text"][:1500])


SOURCE 1
Similarity: 0.6855
Case Information: judgment amarendra nath sen, j. 1.


Text:
out at length the facts and circumstances of this case.or the various submissions which had been made. the decision of the high court meets with our approval and we see no reason to interfere with the judgment and order passed by the high court. the appeal is accordingly dismissed with no order as to costs.

SOURCE 2
Similarity: 0.6847
Case Information: petitioner mamad hassam bhagad and others vs. respondent state of gujarat and others date of judgment 09/05/1996 bench venkataswami k.  j  bench venkataswami k.  j  anand, a.s.  
j  citation 1996 air 2057 jt 1996  5  327 1996 scale  4 313 act headnote judgment j u d g m e n t k. venkataswami,j. this appeal preferred under section 19 of the terrorist and disruptive activities  prevention  act.
1987  hereinafter referred to as tada  challenges the judgment and order of the learned designated judge  tada , jamnagar dated july 1, 1994.
the learned desi

In [46]:
question = """
What was the final decision or outcome of the court?
What did the court ultimately hold or order?
Was the case allowed, dismissed, disposed of, or otherwise decided?
"""

answer, sources = generate_answer(
    question,
    top_k=10
)

print(answer)

Trying model: gemini-3.5-flash
gemini-3.5-flash is temporarily unavailable. Trying next model...
Trying model: gemini-3.5-flash-lite
Successfully generated using: gemini-3.5-flash-lite
Based on the provided retrieved legal judgment context, different cases resulted in different outcomes:

* **Dismissed:** Several appeals were dismissed by the court (Sources 1, 3, 7, 8, and 10). Specifically, some appeals found no merit and were dismissed (Sources 1, 3, and 8), one appeal failed and was dismissed (Source 7), and another resulted in the court finding no ground to interfere with the High Court's order, leading to the dismissal of the appeals (Source 10).
* **Allowed:** Certain appeals were allowed, setting aside the previous orders of the High Court and appellate court while confirming the trial court's order (Source 5), or setting aside the orders rejecting the application under order 9 rule 13 and directing the trial court to hear and dispose of the application in accordance with law, w

In [47]:
# Add Similarity Threshold
def retrieve_documents(query, top_k=5, threshold=0.30):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if score >= threshold:

            result = chunks_df.iloc[idx].to_dict()

            result["similarity_score"] = float(score)

            results.append(result)

    return results

In [48]:
# Evaluate Retrieval
evaluation_questions = [
    "What was the main legal issue considered by the court?",
    "What was the final decision?",
    "What arguments were presented before the court?",
    "What reasoning did the court provide?",
    "What relief was granted by the court?"
]

In [49]:
for question in evaluation_questions:

    results = retrieve_documents(
        question,
        top_k=5
    )

    print("\nQuestion:")
    print(question)

    print("Retrieved documents:",
          len(results))

    if results:
        print(
            "Top similarity:",
            round(
                results[0]["similarity_score"],
                4
            )
        )


Question:
What was the main legal issue considered by the court?
Retrieved documents: 5
Top similarity: 0.6287

Question:
What was the final decision?
Retrieved documents: 5
Top similarity: 0.5172

Question:
What arguments were presented before the court?
Retrieved documents: 5
Top similarity: 0.6312

Question:
What reasoning did the court provide?
Retrieved documents: 5
Top similarity: 0.6463

Question:
What relief was granted by the court?
Retrieved documents: 5
Top similarity: 0.658


In [50]:
# Create Evaluation Dataset
evaluation_data = [
    {
        "question": "What was the main legal issue?",
        "expected": "Main issue discussed in the judgment"
    },
    {
        "question": "What was the final decision?",
        "expected": "Final decision of the court"
    },
    {
        "question": "What reasoning did the court provide?",
        "expected": "Court's reasoning"
    },
    {
        "question": "What relief was granted?",
        "expected": "Relief granted by court"
    }
]

In [51]:
import os

os.makedirs("rag_artifacts", exist_ok=True)

faiss.write_index(
    index,
    "rag_artifacts/legal_judgment_faiss.index"
)

chunks_df.to_pickle(
    "rag_artifacts/legal_judgment_chunks.pkl"
)

print("RAG artifacts saved successfully.")

RAG artifacts saved successfully.


# RAG Retrieval Evaluation

In [52]:

# RAG EVALUATION QUESTIONS

evaluation_questions = [
    "What was the main legal issue considered by the court?",
    "What was the final decision or outcome of the court?",
    "What reasoning did the court provide for its decision?",
    "What arguments were presented before the court?",
    "What relief was granted by the court?",
    "What legal provision was relevant to the case?",
    "What was the court's final order?",
    "What did the court ultimately hold?"
]

print("Total evaluation questions:",
      len(evaluation_questions))

Total evaluation questions: 8


# Measure Retrieval Quality

In [53]:

# RETRIEVAL EVALUATION


evaluation_results = []

for question in evaluation_questions:

    results = retrieve_documents(
        question,
        top_k=10
    )

    if len(results) > 0:

        top_1_score = results[0]["similarity_score"]

        top_5_scores = [
            r["similarity_score"]
            for r in results[:5]
        ]

        top_10_scores = [
            r["similarity_score"]
            for r in results[:10]
        ]

        evaluation_results.append({
            "question": question,
            "top_1_similarity": top_1_score,
            "avg_top_5_similarity": np.mean(top_5_scores),
            "avg_top_10_similarity": np.mean(top_10_scores),
            "retrieved_documents": len(results)
        })

    else:

        evaluation_results.append({
            "question": question,
            "top_1_similarity": 0,
            "avg_top_5_similarity": 0,
            "avg_top_10_similarity": 0,
            "retrieved_documents": 0
        })


evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,question,top_1_similarity,avg_top_5_similarity,avg_top_10_similarity,retrieved_documents
0,What was the main legal issue considered by th...,0.628708,0.609452,0.601982,10
1,What was the final decision or outcome of the ...,0.694777,0.658445,0.638548,10
2,What reasoning did the court provide for its d...,0.642099,0.621291,0.606790,10
3,What arguments were presented before the court?,0.631213,0.610339,0.603093,10
4,What relief was granted by the court?,0.658048,0.635227,0.621308,10
5,What legal provision was relevant to the case?,0.641855,0.634689,0.627383,10
6,What was the court's final order?,0.682260,0.659806,0.647567,10
7,What did the court ultimately hold?,0.640677,0.610234,0.592769,10


In [54]:
# Overall Retrieval Statistics
print("Average Top-1 Similarity:",
      round(
          evaluation_df["top_1_similarity"].mean(),
          4
      ))

print(
    "Average Top-5 Similarity:",
    round(
        evaluation_df["avg_top_5_similarity"].mean(),
        4
    )
)

print(
    "Average Top-10 Similarity:",
    round(
        evaluation_df["avg_top_10_similarity"].mean(),
        4
    )
)

Average Top-1 Similarity: 0.6525
Average Top-5 Similarity: 0.6299
Average Top-10 Similarity: 0.6174


# Retrieval Results

In [55]:
for _, row in evaluation_df.iterrows():

    print("\n" + "=" * 90)

    print("Question:")
    print(row["question"])

    print(
        "Top-1 Similarity:",
        round(row["top_1_similarity"], 4)
    )

    print(
        "Average Top-5:",
        round(row["avg_top_5_similarity"], 4)
    )

    print(
        "Average Top-10:",
        round(row["avg_top_10_similarity"], 4)
    )


Question:
What was the main legal issue considered by the court?
Top-1 Similarity: 0.6287
Average Top-5: 0.6095
Average Top-10: 0.602

Question:
What was the final decision or outcome of the court?
Top-1 Similarity: 0.6948
Average Top-5: 0.6584
Average Top-10: 0.6385

Question:
What reasoning did the court provide for its decision?
Top-1 Similarity: 0.6421
Average Top-5: 0.6213
Average Top-10: 0.6068

Question:
What arguments were presented before the court?
Top-1 Similarity: 0.6312
Average Top-5: 0.6103
Average Top-10: 0.6031

Question:
What relief was granted by the court?
Top-1 Similarity: 0.658
Average Top-5: 0.6352
Average Top-10: 0.6213

Question:
What legal provision was relevant to the case?
Top-1 Similarity: 0.6419
Average Top-5: 0.6347
Average Top-10: 0.6274

Question:
What was the court's final order?
Top-1 Similarity: 0.6823
Average Top-5: 0.6598
Average Top-10: 0.6476

Question:
What did the court ultimately hold?
Top-1 Similarity: 0.6407
Average Top-5: 0.6102
Average Top

# RAG Answer Evaluation

In [56]:

# GENERATE RAG ANSWERS FOR EVALUATION

rag_evaluation = []

for question in evaluation_questions:

    print("\nProcessing:")
    print(question)

    answer, sources = generate_answer(
        question,
        top_k=10
    )

    rag_evaluation.append({
        "question": question,
        "answer": answer,
        "number_of_sources": len(sources)
    })

print("\nEvaluation completed.")


Processing:
What was the main legal issue considered by the court?
Trying model: gemini-3.5-flash
gemini-3.5-flash is temporarily unavailable. Trying next model...
Trying model: gemini-3.5-flash-lite
Successfully generated using: gemini-3.5-flash-lite

Processing:
What was the final decision or outcome of the court?
Trying model: gemini-3.5-flash
Successfully generated using: gemini-3.5-flash

Processing:
What reasoning did the court provide for its decision?
Trying model: gemini-3.5-flash
gemini-3.5-flash is temporarily unavailable. Trying next model...
Trying model: gemini-3.5-flash-lite
Successfully generated using: gemini-3.5-flash-lite

Processing:
What arguments were presented before the court?
Trying model: gemini-3.5-flash
gemini-3.5-flash is temporarily unavailable. Trying next model...
Trying model: gemini-3.5-flash-lite
Successfully generated using: gemini-3.5-flash-lite

Processing:
What relief was granted by the court?
Trying model: gemini-3.5-flash
gemini-3.5-flash is te

In [57]:
rag_evaluation_df = pd.DataFrame(
    rag_evaluation
)

rag_evaluation_df

,question,answer,number_of_sources
0,What was the main legal issue considered by th...,Insufficient information in the retrieved docu...,10
1,What was the final decision or outcome of the ...,Insufficient information in the retrieved docu...,10
2,What reasoning did the court provide for its d...,Insufficient information in the retrieved docu...,10
3,What arguments were presented before the court?,"Based on the provided retrieved documents, var...",10
4,What relief was granted by the court?,Insufficient information in the retrieved docu...,10
5,What legal provision was relevant to the case?,"Based on the provided retrieved context, multi...",10
6,What was the court's final order?,"Based on the retrieved legal documents, there ...",10
7,What did the court ultimately hold?,Insufficient information in the retrieved docu...,10


In [58]:
rag_evaluation_df.to_csv(
    "rag_evaluation_results.csv",
    index=False
)

evaluation_df.to_csv(
    "retrieval_evaluation_results.csv",
    index=False
)

print("Evaluation files saved successfully.")

Evaluation files saved successfully.


In [59]:
def expand_legal_query(question):

    query_expansions = {

        "final decision": """
        final decision final order court held
        judgment outcome disposed dismissed allowed
        relief granted order of the court
        """,

        "main legal issue": """
        main legal issue issue before the court
        question of law legal question
        """,

        "reasoning": """
        reasoning court observed court held
        rationale findings legal reasoning
        """,

        "relief": """
        relief granted remedy order direction
        final relief granted by the court
        """
    }

    expanded_query = question

    question_lower = question.lower()

    for keyword, expansion in query_expansions.items():

        if keyword in question_lower:

            expanded_query += " " + expansion

    return expanded_query

In [60]:
# Improved Retrieval Function
def retrieve_legal_documents(
    question,
    top_k=10
):

    # Expand legal query
    expanded_query = expand_legal_query(
        question
    )

    # Retrieve documents
    results = retrieve_documents(
        expanded_query,
        top_k=top_k
    )

    return results

In [61]:
question = """
What was the final decision of the court?
"""

results = retrieve_legal_documents(
    question,
    top_k=10
)

for i, result in enumerate(results):

    print("\n" + "=" * 90)

    print("SOURCE:", i + 1)

    print(
        "Similarity:",
        round(
            result["similarity_score"],
            4
        )
    )

    print(
        "Case Information:",
        result["case_info"]
    )

    print("\nText:")
    print(result["text"][:1500])


SOURCE: 1
Similarity: 0.7111
Case Information: petitioner mamad hassam bhagad and others vs. respondent state of gujarat and others date of judgment 09/05/1996 bench venkataswami k.  j  bench venkataswami k.  j  anand, a.s.  
j  citation 1996 air 2057 jt 1996  5  327 1996 scale  4 313 act headnote judgment j u d g m e n t k. venkataswami,j. this appeal preferred under section 19 of the terrorist and disruptive activities  prevention  act.
1987  hereinafter referred to as tada  challenges the judgment and order of the learned designated judge  tada , jamnagar dated july 1, 1994.
the learned designated judge confirmed an order of attachment of seven vessels ships under section 7a of tada pending further investigation in the matter.
at the outset we would like to make it clear that we do not propose to deal with the matter either in detail or to give a final decision in this matter in view of the fact that when the judgment under appeal was made the matter was under the investigation sta

In [63]:

# SEARCH FOR A SPECIFIC CASE


case_name = "Kikiben Ors vs Balvantlal Shivlal"

# Search case information directly
case_matches = chunks_df[
    chunks_df["case_info"]
    .str.contains(
        "Kikiben",
        case=False,
        na=False
    )
]

print("Matching chunks:", len(case_matches))

case_matches.head()

Matching chunks: 2


,case_id,chunk_id,case_category,case_type,case_info,text
38072,7064,0,civil,appeal,\nsupreme court of india\nkikiben ors vs balva...,Case Category: civil Case Type: appeal Case In...
38073,7064,1,civil,appeal,\nsupreme court of india\nkikiben ors vs balva...,not correct and is contrary to the decision of...


In [64]:
# Display the matching case information

for i, row in case_matches.head(5).iterrows():

    print("=" * 100)

    print("Case Information:")
    print(row["case_info"])

    print("\nJudgment Text:")
    print(row["text"][:2000])

    print()

Case Information:

supreme court of india
kikiben ors vs balvantlal shivlal  dead  by lrs.
 on 15 september, 1999equivalent citations 2000  1  scale 357
bench k venkataswami, d wadhwa
order

1.
application for substitution is allowed.


2.


Judgment Text:
Case Category: civil Case Type: appeal Case Information: supreme court of india kikiben ors vs balvantlal shivlal dead by lrs. on 15 september, 1999equivalent citations 2000 1 scale 357 bench k venkataswami, d wadhwa order 1. application for substitution is allowed. 2. Judgment: these appeals are preferred against the common judgment and order of the gujarat high court in first appeal numbers 387/77 and 388/77 dated 12.10.1984. the appellants are the defendants in the original suit. the respondents-plaintiffs preferred spl. civil suits 131/74 132/74 for enforcement of a testamentary term in their favour and against the appellants defendants herein. the suits were decreed on 31.1.1977. 3. the case of the appellants defendants was that

In [65]:
def retrieve_case_specific_documents(
    case_name,
    question,
    top_k=10
):


    # Find chunks belonging to the requested case

    case_keyword = case_name.split()[0]

    case_matches = chunks_df[
        chunks_df["case_info"]
        .str.contains(
            case_keyword,
            case=False,
            na=False
        )
    ].copy()

    print(
        "Case-related chunks found:",
        len(case_matches)
    )

    # If case not found


    if len(case_matches) == 0:

        return []


    # Embed the question


    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )


    # Embed only the case chunks


    case_embeddings = embedding_model.encode(
        case_matches["text"].tolist(),
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # Calculate similarity


    scores = np.dot(
        case_embeddings,
        query_embedding[0]
    )

    case_matches["similarity_score"] = scores


    # Sort by relevance


    case_matches = case_matches.sort_values(
        "similarity_score",
        ascending=False
    )


    # Return Top-K


    return case_matches.head(top_k).to_dict(
        orient="records"
    )

In [66]:
question = """
What was the final decision or order of the court?
"""

sources = retrieve_case_specific_documents(
    "Kikiben Ors vs Balvantlal Shivlal",
    question,
    top_k=5
)

for i, source in enumerate(sources):

    print("\n" + "=" * 100)

    print("SOURCE:", i + 1)

    print(
        "Similarity:",
        round(
            source["similarity_score"],
            4
        )
    )

    print(
        "Case Information:",
        source["case_info"]
    )

    print("\nText:")
    print(source["text"][:2000])

Case-related chunks found: 2

SOURCE: 1
Similarity: 0.5769
Case Information: 
supreme court of india
kikiben ors vs balvantlal shivlal  dead  by lrs.
 on 15 september, 1999equivalent citations 2000  1  scale 357
bench k venkataswami, d wadhwa
order

1.
application for substitution is allowed.


2.


Text:
not correct and is contrary to the decision of this court as noticed above. this position is also not disputed. 7. in the circumstances, we are of the view that the high court was not right in dismissing the appeals by confirming the decree of the trial court. the appeals are allowed. suits filed by the respondents stand dismissed. however, there will be no order as to costs. the interim applications are accordingly disposed of.

SOURCE: 2
Similarity: 0.4361
Case Information: 
supreme court of india
kikiben ors vs balvantlal shivlal  dead  by lrs.
 on 15 september, 1999equivalent citations 2000  1  scale 357
bench k venkataswami, d wadhwa
order

1.
application for substitution is allo

In [67]:
def generate_case_answer(
    case_name,
    question,
    top_k=5
):

    # -----------------------------------
    # Retrieve case-specific documents
    # -----------------------------------

    sources = retrieve_case_specific_documents(
        case_name,
        question,
        top_k=top_k
    )

    if len(sources) == 0:

        return (
            "The requested case could not be found "
            "in the retrieved dataset.",
            []
        )

    # -----------------------------------
    # Build context
    # -----------------------------------

    context_parts = []

    for i, source in enumerate(sources):

        context_parts.append(
            f"""
SOURCE {i + 1}

Case:
{source["case_info"]}

Relevant Judgment:
{source["text"]}
"""
        )

    context = "\n".join(context_parts)

    # -----------------------------------
    # Legal RAG prompt
    # -----------------------------------

    prompt = f"""
You are a legal document retrieval assistant.

The user is asking about this specific Indian
Supreme Court case:

{case_name}

Answer the question ONLY using the retrieved
judgment excerpts below.

Do not use outside knowledge.

If the retrieved excerpts contain the final
decision, clearly state what the court decided.

Question:
{question}

Retrieved Judgment Context:
{context}

Answer:
"""

    # -----------------------------------
    # Gemini models with fallback
    # -----------------------------------

    models = [
        "gemini-3.5-flash",
        "gemini-3.5-flash-lite",
        "gemini-2.5-flash-lite"
    ]

    for model_name in models:

        try:

            print(
                f"Trying model: {model_name}"
            )

            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            print(
                f"Successfully generated using: "
                f"{model_name}"
            )

            return response.text, sources

        except ServerError:

            print(
                f"{model_name} unavailable. "
                f"Trying next model..."
            )

            time.sleep(2)

    return (
        "Gemini models are temporarily unavailable.",
        sources
    )

In [68]:
case_name = "Kikiben Ors vs Balvantlal Shivlal"

question = """
What was the final decision of the court?
"""

answer, sources = generate_case_answer(
    case_name,
    question,
    top_k=5
)

print("\nFINAL ANSWER")
print("=" * 80)
print(answer)

Case-related chunks found: 2
Trying model: gemini-3.5-flash
gemini-3.5-flash unavailable. Trying next model...
Trying model: gemini-3.5-flash-lite
Successfully generated using: gemini-3.5-flash-lite

FINAL ANSWER
Based on the retrieved judgment excerpts, the final decision of the court was:

* The application for substitution was allowed.
* The appeals were allowed, and the High Court's judgment confirming the trial court's decree was set aside. 
* The suits filed by the respondents stand dismissed.
* There was no order as to costs.


# Final RAG Pipeline

In [69]:

# FINAL LEGAL RAG PIPELINE


def legal_rag_pipeline(
    case_name,
    question,
    top_k=5
):
    """
    Complete case-specific Legal RAG pipeline.

    Flow:
    Case Name
        ↓
    Case-specific Retrieval
        ↓
    Relevant Context
        ↓
    Gemini LLM
        ↓
    Grounded Answer + Sources
    """

    # Retrieve relevant case chunks


    sources = retrieve_case_specific_documents(
        case_name,
        question,
        top_k=top_k
    )

    # Handle missing case


    if len(sources) == 0:

        return {
            "answer": (
                "The requested case could not be found "
                "in the retrieved dataset."
            ),
            "sources": []
        }

    # Build context

    context_parts = []

    for i, source in enumerate(sources):

        context_parts.append(
            f"""
SOURCE {i + 1}

Case Information:
{source["case_info"]}

Relevant Judgment:
{source["text"]}
"""
        )

    context = "\n".join(context_parts)

    # Create grounded legal prompt

    prompt = f"""
You are a legal document retrieval assistant.

You must answer the user's question using ONLY
the retrieved judgment excerpts provided below.

Do not use outside knowledge.

Do not invent legal facts.

If the retrieved context does not contain enough
information, clearly state:

"Insufficient information in the retrieved documents."

Case:
{case_name}

Question:
{question}

Retrieved Judgment Context:
{context}

Provide a concise and factual answer.

Answer:
"""


    # Gemini model fallback

    models = [
        "gemini-3.5-flash",
        "gemini-3.5-flash-lite",
        "gemini-2.5-flash-lite"
    ]

    for model_name in models:

        try:

            print(
                f"Trying model: {model_name}"
            )

            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            print(
                f"Successfully generated using: "
                f"{model_name}"
            )

            return {
                "answer": response.text,
                "sources": sources,
                "model": model_name
            }

        except ServerError:

            print(
                f"{model_name} unavailable. "
                f"Trying next model..."
            )

            time.sleep(2)

    # All models unavailable

    return {
        "answer": (
            "Gemini models are temporarily unavailable. "
            "Please try again later."
        ),
        "sources": sources,
        "model": None
    }

# Test Final Pipeline

In [70]:

# FINAL PIPELINE TEST

case_name = "Kikiben Ors vs Balvantlal Shivlal"

question = """
What was the final decision of the court?
"""

result = legal_rag_pipeline(
    case_name=case_name,
    question=question,
    top_k=5
)

print("\nFINAL ANSWER")
print("=" * 80)
print(result["answer"])

print("\nMODEL USED")
print("=" * 80)
print(result["model"])

print("\nRETRIEVED SOURCES")
print("=" * 80)

for i, source in enumerate(result["sources"]):

    print(f"\nSource {i + 1}")

    print(
        "Similarity:",
        round(
            source["similarity_score"],
            4
        )
    )

    print(
        "Case:",
        source["case_info"]
    )

Case-related chunks found: 2
Trying model: gemini-3.5-flash
Successfully generated using: gemini-3.5-flash

FINAL ANSWER
Based on the retrieved documents, the final decision of the court was as follows:

1. The application for substitution is allowed.
2. The appeals are allowed.
3. The suits filed by the respondents stand dismissed.
4. There will be no order as to costs.
5. The interim applications are disposed of accordingly.

MODEL USED
gemini-3.5-flash

RETRIEVED SOURCES

Source 1
Similarity: 0.5846
Case: 
supreme court of india
kikiben ors vs balvantlal shivlal  dead  by lrs.
 on 15 september, 1999equivalent citations 2000  1  scale 357
bench k venkataswami, d wadhwa
order

1.
application for substitution is allowed.


2.


Source 2
Similarity: 0.4397
Case: 
supreme court of india
kikiben ors vs balvantlal shivlal  dead  by lrs.
 on 15 september, 1999equivalent citations 2000  1  scale 357
bench k venkataswami, d wadhwa
order

1.
application for substitution is allowed.


2.



# Test Multiple Questions

In [71]:
# ==========================================
# MULTIPLE QUESTION TEST
# ==========================================

case_name = "Kikiben Ors vs Balvantlal Shivlal"

test_questions = [
    "What was the final decision of the court?",
    "What was the main legal issue in this case?",
    "What did the court decide regarding the appeals?",
    "What happened to the suits filed by the respondents?",
    "Did the court award any costs?"
]

for question in test_questions:

    print("\n")
    print("=" * 100)
    print("QUESTION:")
    print(question)
    print("=" * 100)

    result = legal_rag_pipeline(
        case_name,
        question,
        top_k=5
    )

    print("\nANSWER:")
    print(result["answer"])

    print(
        "\nMODEL:",
        result["model"]
    )



QUESTION:
What was the final decision of the court?
Case-related chunks found: 2
Trying model: gemini-3.5-flash
gemini-3.5-flash unavailable. Trying next model...
Trying model: gemini-3.5-flash-lite
Successfully generated using: gemini-3.5-flash-lite

ANSWER:
Based on the retrieved documents, the final decision of the court was:

* The application for substitution is allowed.
* The appeals are allowed.
* The suits filed by the respondents stand dismissed.
* There will be no order as to costs.
* The interim applications are disposed of accordingly.

MODEL: gemini-3.5-flash-lite


QUESTION:
What was the main legal issue in this case?
Case-related chunks found: 2
Trying model: gemini-3.5-flash
gemini-3.5-flash unavailable. Trying next model...
Trying model: gemini-3.5-flash-lite
Successfully generated using: gemini-3.5-flash-lite

ANSWER:
Based on the retrieved documents, the main legal issue involved whether Section 14(1) or Section 14(2) of the Hindu Succession Act, 1956, applied to t

# Create a Simple Evaluation Table

In [74]:

# RETRIEVAL-ONLY EVALUATION

retrieval_results = []

case_name = "Kikiben Ors vs Balvantlal Shivlal"

for question in test_questions:

    sources = retrieve_case_specific_documents(
        case_name,
        question,
        top_k=5
    )

    if len(sources) > 0:

        top_similarity = sources[0]["similarity_score"]

        average_similarity = np.mean([
            source["similarity_score"]
            for source in sources
        ])

    else:

        top_similarity = 0
        average_similarity = 0

    retrieval_results.append({
        "case": case_name,
        "question": question,
        "sources_retrieved": len(sources),
        "top_similarity": top_similarity,
        "average_similarity": average_similarity
    })


retrieval_eval_df = pd.DataFrame(
    retrieval_results
)

retrieval_eval_df

Case-related chunks found: 2
Case-related chunks found: 2
Case-related chunks found: 2
Case-related chunks found: 2
Case-related chunks found: 2


,case,question,sources_retrieved,top_similarity,average_similarity
0,Kikiben Ors vs Balvantlal Shivlal,What was the final decision of the court?,2,0.584620,0.512183
1,Kikiben Ors vs Balvantlal Shivlal,What was the main legal issue in this case?,2,0.420693,0.394241
2,Kikiben Ors vs Balvantlal Shivlal,What did the court decide regarding the appeals?,2,0.640997,0.572465
3,Kikiben Ors vs Balvantlal Shivlal,What happened to the suits filed by the respon...,2,0.543400,0.457179
4,Kikiben Ors vs Balvantlal Shivlal,Did the court award any costs?,2,0.525217,0.439615


In [75]:
print(
    "Average Top Similarity:",
    round(
        retrieval_eval_df["top_similarity"].mean(),
        4
    )
)

print(
    "Average Similarity:",
    round(
        retrieval_eval_df["average_similarity"].mean(),
        4
    )
)

print(
    "Average Sources Retrieved:",
    round(
        retrieval_eval_df["sources_retrieved"].mean(),
        2
    )
)

Average Top Similarity: 0.543
Average Similarity: 0.4751
Average Sources Retrieved: 2.0


In [76]:
retrieval_eval_df.to_csv(
    "retrieval_only_evaluation.csv",
    index=False
)

print("Retrieval evaluation saved successfully.")

Retrieval evaluation saved successfully.


In [78]:

# VERIFY FINAL RAG ARTIFACTS


import os

required_files = [
    "rag_artifacts/legal_judgment_faiss.index",
    "rag_artifacts/legal_judgment_chunks.pkl",
    "retrieval_only_evaluation.csv"
]

print("Checking required files...\n")

for file_path in required_files:

    if os.path.exists(file_path):

        size_mb = os.path.getsize(file_path) / (1024 * 1024)

        print(f"✅ {file_path}")
        print(f"   Size: {size_mb:.2f} MB")

    else:

        print(f"❌ Missing: {file_path}")

Checking required files...

✅ rag_artifacts/legal_judgment_faiss.index
   Size: 78.75 MB
✅ rag_artifacts/legal_judgment_chunks.pkl
   Size: 201.75 MB
✅ retrieval_only_evaluation.csv
   Size: 0.00 MB


# Deployment files prepare

In [79]:

# CREATE DEPLOYMENT PACKAGE

import os
import shutil

deployment_folder = "legal_rag_deployment"

os.makedirs(
    deployment_folder,
    exist_ok=True
)

# Copy FAISS index
shutil.copy2(
    "rag_artifacts/legal_judgment_faiss.index",
    f"{deployment_folder}/legal_judgment_faiss.index"
)

# Copy chunk metadata
shutil.copy2(
    "rag_artifacts/legal_judgment_chunks.pkl",
    f"{deployment_folder}/legal_judgment_chunks.pkl"
)

print("Deployment files created successfully.")

print("\nFiles:")

for file in os.listdir(deployment_folder):

    path = os.path.join(
        deployment_folder,
        file
    )

    size_mb = os.path.getsize(path) / (1024 * 1024)

    print(
        f"{file} → {size_mb:.2f} MB"
    )

Deployment files created successfully.

Files:
legal_judgment_faiss.index → 78.75 MB
legal_judgment_chunks.pkl → 201.75 MB
